# Operations Agent Demo

The Operations/Logistics Agent is the fourth specialist, now registered alongside Finance,
Sentiment, and Sales in the boss agent's roster (`backend/agents/boss/registry.py`).

## Its 7 tools

| Tool | Computes | Data reality |
|---|---|---|
| `calculate_delivery_delay` | Avg delay + late-delivery rate vs. Olist's own estimate | Computed directly |
| `seller_performance_score` | Seller reliability: on-time %, avg delay | Reliability lens only — deliberately overlaps with Sales' `seller_sales_ranking` (revenue/volume), a different lens on the same sellers per CLAUDE.md |
| `flag_late_shipments` | Mild vs. severe lateness buckets | Computed directly |
| `shipping_cost_analysis` | Freight cost, interstate vs. intrastate | Logistics-efficiency framing — deliberately distinct from Finance's margin-impact framing of the same `freight_value` field |
| `carrier_performance_comparison` | — | **Explicitly returns "not available"** — Olist's schema has no carrier/shipping-company identifier at all, so there's nothing to compare across carriers |
| `fulfillment_bottleneck_detection` | Avg duration of each fulfillment stage | Uses fractional-day precision (not whole-day `datediff`) so same-day stages like approval still show meaningful averages |
| `estimated_vs_actual_delivery_accuracy` | MAE + bias of Olist's own delivery estimate | Bias sign is a real signal: negative means Olist under-promises and over-delivers (conservative estimate, arrives early) |

## The `carrier_performance_comparison` gap

This is the third tool across all specialists so far (after Finance's `calculate_cogs` and — in
spirit — Sales' funnel proxy) that hits a hard wall in what Olist's schema can support. There is
genuinely no field anywhere in this dataset identifying which carrier fulfilled a shipment.
Rather than approximate this with something misleading, the tool returns
`{"available": False, "reason": ...}` — consistent with the project's data-honesty pattern
established from the very first specialist agent.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from backend.agents.operations import OperationsAgent
from backend.db import DatabricksClient

db = DatabricksClient()
agent = OperationsAgent(db)

## Run the agent against live Delta tables

In [2]:
briefing = agent.run("How is delivery performance looking?")

print(f"Agent: {briefing.agent.value}")
print(f"Execution time: {briefing.execution_time_ms:.0f}ms")
print(f"Tool calls: {len(briefing.tool_calls)} ({sum(1 for t in briefing.tool_calls if t.success)} succeeded)")

Agent: Operations
Execution time: 6734ms
Tool calls: 7 (7 succeeded)


## Inspect the findings

In [3]:
for f in briefing.findings:
    print(f"[{f.severity.upper()}] (confidence {f.confidence:.2f}) {f.claim}")
    print(f"  source: {f.source}\n")

[INFO] (confidence 0.90) 8.11% of deliveries arrive late (avg delay -11.88 days) out of 96470 delivered orders
  source: calculate_delivery_delay

[WARNING] (confidence 0.85) Least reliable seller (min 5 orders) is 633ecdf879b94b5337cca303328e4a25 at 33.33% on-time
  source: seller_performance_score

[WARNING] (confidence 0.90) 2.97% of deliveries are severely late (>7 days), 3.81% mildly late
  source: flag_late_shipments

[INFO] (confidence 0.85) Interstate shipments average 23.69 freight vs 13.46 for intrastate
  source: shipping_cost_analysis

[WARNING] (confidence 1.00) Carrier performance cannot be compared - no carrier identifier exists in the source data
  source: carrier_performance_comparison

[INFO] (confidence 0.85) The slowest fulfillment stage is 'transit (carrier pickup -> customer)' at 9.33 days on average
  source: fulfillment_bottleneck_detection

[INFO] (confidence 0.90) Delivery estimate accuracy: mean absolute error 13.31 days, bias -11.88 days (early-leaning)
  so

## Through the boss agent

Four specialists now registered. A delivery-and-satisfaction query should pull in Operations
and Sentiment together — worth checking whether late deliveries and negative reviews line up,
or whether the boss agent finds a disconnect worth flagging as dissent.

In [4]:
from backend.agents.boss import BossAgent

boss = BossAgent(db)
rec = boss.run("Are delivery problems hurting customer satisfaction?")

print(f"Agents invoked: {[a.value for a in rec.agents_invoked]}")
print(f"Overall confidence: {rec.confidence_overall}")
print(f"\nSynthesis:\n{rec.synthesis}")

if rec.dissents:
    print("\nDissents:")
    for d in rec.dissents:
        print(f"  - {[a.value for a in d.agents_involved]}: {d.summary}")

Agents invoked: ['Operations', 'Sentiment']
Overall confidence: 0.78

Synthesis:
Operations’s calculate_delivery_delay tool found that 8.11% of 96,470 delivered orders arrive late, with an average delay of –11.88 days (bias) and a mean absolute error of 13.31 days in delivery estimates (Operations, estimated_vs_actual_delivery_accuracy). The transit stage from carrier pickup to customer is the slowest, averaging 9.33 days (Operations, fulfillment_bottleneck_detection). Severe lateness (>7 days) accounts for 2.97% of deliveries, while mildly late deliveries (3.81%) also contribute to overall delay (Operations, flag_late_shipments). Shipping costs are higher for interstate shipments (23.69) versus intrastate (13.46) (Operations, shipping_cost_analysis). No carrier identifiers exist, preventing performance comparison (Operations, carrier_performance_comparison). Sentiment’s search_reviews tool returned no reviews containing the phrase “Are delivery problems hurting customer satisfaction?”

## What's pending

- Growth, Risk, Compliance/HR — 3 specialists remaining
- `seller_performance_score` (Ops, reliability) vs. `seller_sales_ranking` (Sales, volume) is a clean setup for the boss agent to cross-reference "top seller by volume is also your worst on delivery" once a query prompts it — CLAUDE.md calls this out explicitly as a strong demo moment